# QC walkthrough (scanpy/scverse)

Use this notebook to mirror the CLI QC flow: load AnnData, annotate QC metrics, apply basic thresholds, and export filtered data + figures. Defaults follow the scverse/best-practices tutorial values.

In [ ]:
from pathlib import Path
import scanpy as sc

from vcc.data import load_anndata, load_dataset
from vcc.qc import (
    QCThresholds,
    annotate_qc_metrics,
    cell_filter_mask,
    filter_genes,
    plot_basic_qc,
    summarize_qc,
)

# Point to your dataset. If adata_path is None, the named dataset is loaded via VCC_DATA_DIR.
adata_path = None  # Path("/Users/saberhq/code/vcc/vcc_data/adata_Training.h5ad")
dataset_key = "train"
data_dir = None  # Path("/Users/saberhq/code/vcc/vcc_data")

In [ ]:
if adata_path is not None:
    adata = load_anndata(adata_path)
else:
    adata = load_dataset(dataset_key, data_dir=data_dir)

adata = adata.copy()  # work on a copy to keep the source untouched
print(f"Loaded {adata.n_obs:,} cells × {adata.n_vars:,} genes")

In [ ]:
annotate_qc_metrics(adata)
qc_thresholds = QCThresholds()  # defaults from scverse/best-practices tutorials
genes_removed = filter_genes(adata, min_cells=qc_thresholds.min_cells_per_gene)
mask = cell_filter_mask(adata, qc_thresholds)
adata_filtered = adata[mask].copy()
summary = summarize_qc(adata, adata_filtered, thresholds=qc_thresholds, genes_dropped=genes_removed)
summary

In [ ]:
fig_paths = plot_basic_qc(adata, outdir=Path("qc_figures"))
fig_paths

In [ ]:
filtered_path = Path("adata_qc_filtered.h5ad")
adata_filtered.write(filtered_path)
filtered_path